In [1]:
%%capture

import re

import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
%pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
%pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
%pip install --no-deps --upgrade "torchao>=0.16.0"
%pip install --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
%pip install torchcodec

import torch; torch._dynamo.config.recompile_limit = 64;

%pip install --no-deps --upgrade timm  # For Gemma 4 vision/audio

## dataset

In [3]:
%%capture

%pip install gdown datasets

import os
import gdown
import zipfile

In [4]:
file_id = "1WhOgsEWmLSnEG2-K8R2n_hogFLGPkI8I"
download_url = f"https://drive.google.com/uc?id={file_id}"

zip_path = "/content/mredditsum.zip"
extract_dir = "/content/mredditsum_dataset"

# Download if not already downloaded
if not os.path.exists(zip_path):
    print("Downloading mRedditSum dataset...")
    gdown.download(download_url, zip_path, quiet=False)
else:
    print("ZIP file already exists. Skipping download.")

# Extract the ZIP file
if not os.path.exists(extract_dir):
    print("Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_dir)
    print("Extraction complete.")
else:
    print("Extracted folder already exists. Skipping extraction.")

print(f"Dataset extracted to: {extract_dir}")

Downloading...
From (original): https://drive.google.com/uc?id=1WhOgsEWmLSnEG2-K8R2n_hogFLGPkI8I
From (redirected): https://drive.google.com/uc?id=1WhOgsEWmLSnEG2-K8R2n_hogFLGPkI8I&confirm=t&uuid=048d8df9-88c1-4ef9-a177-3d663c204d43
To: /content/mredditsum.zip
100%|██████████| 28.3M/28.3M [00:01<00:00, 21.3MB/s]


Extracting files...
Extraction complete.
Dataset extracted to: /content/mredditsum_dataset


In [7]:
extract_dir

'/content/mredditsum_dataset'

In [17]:
import os

# List the top-level contents of the extracted dataset
extract_dir = "/content/mredditsum_dataset"
for root, dirs, files in os.walk(extract_dir):
    print(root, dirs[:10], files[:10])  # only show first 10 items

/content/mredditsum_dataset ['downloadable_data'] []
/content/mredditsum_dataset/downloadable_data ['raw', 'preprocessed'] ['train_processed_imgcap_src.txt', 'test_processed_bestimgcap_src.txt', 'val_processed_src.txt', 'train_processed_tgt.txt', 'val_processed_imgcap_tgt.txt', 'train_processed_imgcap_tgt.txt', 'test_processed_tgt.txt', 'test_processed_bestimgcap_tgt.txt', 'train_processed_src.txt', 'val_processed_imgcap_src.txt']
/content/mredditsum_dataset/downloadable_data/raw [] ['train.json', 'val.json', 'test.json']
/content/mredditsum_dataset/downloadable_data/preprocessed [] ['train_processed_imgcap_src.txt', 'test_processed_bestimgcap_src.txt', 'val_processed_src.txt', 'train_processed_tgt.txt', 'val_processed_imgcap_tgt.txt', 'train_processed_imgcap_tgt.txt', 'test_processed_tgt.txt', 'test_processed_bestimgcap_tgt.txt', 'train_processed_src.txt', 'val_processed_imgcap_src.txt']


In [5]:
!ls -R {extract_dir}

/content/mredditsum_dataset:
downloadable_data

/content/mredditsum_dataset/downloadable_data:
preprocessed			   train_processed_imgcap_tgt.txt
raw				   train_processed_src.txt
test_processed_bestimgcap_src.txt  train_processed_tgt.txt
test_processed_bestimgcap_tgt.txt  val_processed_imgcap_src.txt
test_processed_src.txt		   val_processed_imgcap_tgt.txt
test_processed_tgt.txt		   val_processed_src.txt
train_processed_imgcap_src.txt	   val_processed_tgt.txt

/content/mredditsum_dataset/downloadable_data/preprocessed:
test_processed_bestimgcap_src.txt  train_processed_src.txt
test_processed_bestimgcap_tgt.txt  train_processed_tgt.txt
test_processed_src.txt		   val_processed_imgcap_src.txt
test_processed_tgt.txt		   val_processed_imgcap_tgt.txt
train_processed_imgcap_src.txt	   val_processed_src.txt
train_processed_imgcap_tgt.txt	   val_processed_tgt.txt

/content/mredditsum_dataset/downloadable_data/raw:
test.json  train.json  val.json


In [12]:
from datasets import Dataset, DatasetDict

data_dir = extract_dir
splits = {
    "train": {"src": "train_processed_src.txt", "tgt": "train_processed_tgt.txt"},
    "val":   {"src": "val_processed_src.txt",   "tgt": "val_processed_tgt.txt"},
    "test":  {"src": "test_processed_src.txt",  "tgt": "test_processed_tgt.txt"},
}

dataset_dict = {}
for split_name, files in splits.items():
    src_path = os.path.join(data_dir + '/downloadable_data/preprocessed/', files["src"])
    tgt_path = os.path.join(data_dir + '/downloadable_data/preprocessed/', files["tgt"])
    
    # Read source and target files line by line
    with open(src_path, "r", encoding="utf-8") as f:
        src_lines = f.read().strip().split("\n")
    with open(tgt_path, "r", encoding="utf-8") as f:
        tgt_lines = f.read().strip().split("\n")
    
    # Make sure we have matching pairs
    assert len(src_lines) == len(tgt_lines), f"Mismatch in {split_name}: {len(src_lines)} src vs {len(tgt_lines)} tgt"
    
    # Create a Hugging Face Dataset for this split
    dataset_dict[split_name] = Dataset.from_dict({
        "source": src_lines,
        "target": tgt_lines
    })

# Combine into a single DatasetDict
mredditsum = DatasetDict(dataset_dict)
print(mredditsum)

DatasetDict({
    train: Dataset({
        features: ['source', 'target'],
        num_rows: 2729
    })
    val: Dataset({
        features: ['source', 'target'],
        num_rows: 152
    })
    test: Dataset({
        features: ['source', 'target'],
        num_rows: 152
    })
})


In [15]:
print(mredditsum['train']['source'][0])

coiccq Original Post: Would you put the sectional on the other side?? OP: So I think I may have made this post confusing. What I would be doing is changing the side the chaise is on and putting the couch in the other side of the room. The side they are in now are bay windows and I feel I’m losing almost a foot of space. I understand that a tv with windows behind wouldn’t be ideal. But I don’t usually watch tv during the day anyway User 1: It sounds like you would be losing foot space regardless because the tv stand wouldn’t be able to fully fit in the bay window either. I think it looks nice as is. Maybe look into if there is a console table you could put behind the couch that is designed to fit with a bay window? OP: I’m getting a tv stand that is less wide to fit the space. User 2: Is there space to put a TV stand in the corner by the windows? User 3: You could put some tall plants behind the sofa or find a sofa table that fits between the couch and the windows? It’s hard to tell how

In [ ]:
print(mredditsum['train']['target'][0])

coiccq The OP wanted to know which wall they should place their sectional couch up against in their living room.  Most commenters agreed that the seating should stay by the windows.  One commenter said to get a smaller couch, but OP said they can't.


## train

In [3]:
from unsloth import FastModel

MODEL_NAME = 'unsloth/gemma-4-E2B-it'

model, tokenizer = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    dtype = None,
    max_seq_length = 1024,
    load_in_4bit = True,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/2011 [00:00<?, ?it/s]

In [17]:
from transformers import TextStreamer

def do_gemma_4_inference(messages, max_new_tokens = 128, with_streamer = True):
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt = True,
        tokenize = True,
        return_dict = True,
        return_tensors = "pt",
    ).to("cuda")
    
    if not with_streamer:
        return model.generate(
            **inputs,
            max_new_tokens = max_new_tokens,
            temperature = 1.0, top_p = 0.95, top_k = 64, # recommended by devs
            use_cache = True
        ), inputs
    
    _ = model.generate(
        **inputs,
        max_new_tokens = max_new_tokens,
        temperature = 1.0, top_p = 0.95, top_k = 64, # recommended by devs
        streamer = TextStreamer(tokenizer, skip_prompt = True),
        use_cache = True
    )

In [23]:
from pandas import DataFrame
from datasets import load_dataset, Dataset

dataset = load_dataset("webis/tldr-17", split="train", trust_remote_code=True, streaming=True)
dataset = dataset.take(50)

df = DataFrame(list(dataset))
dataset = Dataset.from_pandas(df)

In [28]:
import evaluate
from tqdm import tqdm

rouge = evaluate.load("rouge")
model.eval()

system_msg = (
    "You are a helpful assistant that writes concise TL;DR summaries "
    "of Reddit posts in one or two sentences."
)

predictions, references = [], []

eval_subset = dataset.select(range(3))

for example in tqdm(eval_subset, desc="Generating summaries"):
    content = example["content"]
    ref = example["summary"]
    
    messages = [
        {
            "role": "system",
            "content": [{
                "type" : "text",
                "text" : system_msg,
            }]
        },
        {
            "role": "user",
            "content": [{
                "type" : "text",
                "text" : f"SUBREDDIT:\n{example["subreddit"]}\nREDDIT POST:\n{example["content"]}",
            }]
        },
        {
            "role": "assistant",
            "content": [{
                "type" : "text",
                "text" : example["summary"],
            }]
        },
    ]
    
    outputs, inputs = do_gemma_4_inference(messages, with_streamer=False)

    pred = tokenizer.decode(
        outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True
    ).strip()
    
    print('>>> pred', pred)
    print('>>> ref', ref)
    
    predictions.append(pred)
    references.append(ref)

results = rouge.compute(
    predictions=predictions,
    references=references,
    use_stemmer=True,
)

for k, v in results.items():
    print(f"  {k}: {v:.4f}")

Generating summaries:  33%|███▎      | 1/3 [00:04<00:08,  4.03s/it]

>>> pred The author argues that moving to a standard time like UTC or UTC+1 would simplify timekeeping systems, as modern technology and efficient lighting make complex seasonal adjustments unnecessary.
>>> ref Shifting seasonal time is no longer worth it.


Generating summaries:  67%|██████▋   | 2/3 [00:07<00:03,  3.41s/it]

>>> pred Art is subjective, and the poster finds some street art styles repetitive, like CLET's use of recurring images.
>>> ref Personal opinions 'n shit.


Generating summaries: 100%|██████████| 3/3 [00:10<00:00,  3.36s/it]

>>> pred The poster finds the Wall Street Journal bland and full of unnecessary jargon, suggesting it targets a specific demographic with overly complicated language.
>>> ref insults and slack ass insight. 
 Wall Street Journal misses on enough counts that not only did i yawn with boredom, i fell asleep trying to read through this crap. 
 It may be the paper for you, but if your in the the market for a new read, i'd at least counsel you on reading the Washington Post instead, due out on stands anytime, or the Globe, which is slated for a weekly release.
  rouge1: 0.0979
  rouge2: 0.0140
  rougeL: 0.0524
  rougeLsum: 0.0730
